<a href="https://colab.research.google.com/github/gulsum135/Datathon2026/blob/main/datathon2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving train.csv to train.csv


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving sample_submission.csv to sample_submission.csv
Saving test_x.csv to test_x.csv


In [ ]:
import os
import pandas as pd

# Bulunduğumuz klasördeki dosyaları listeyelim
print("Yüklenen Dosyalar:", os.listdir())

# Eğer listede train.csv görüyorsan okumayı dene:
try:
    train_df = pd.read_csv('train.csv')
    print("Harika! train.csv başarıyla okundu. Boyut:", train_df.shape)
except Exception as e:
    print("Hata oluştu:", e)

Yüklenen Dosyalar: ['.config', 'test_x.csv', 'sample_submission.csv', 'sample_data']
Hata oluştu: [Errno 2] No such file or directory: 'train.csv'


In [ ]:


from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# datathon klasörünün içindeki dosyaları listeler
klasor_yolu = '/content/drive/MyDrive/datathon'
print("Klasördeki Dosyalar:", os.listdir(klasor_yolu))

Klasördeki Dosyalar: ['train.csv', 'test_x.csv', 'sample_submission.csv']


In [ ]:
import pandas as pd
import os

klasor_yolu = '/content/drive/MyDrive/datathon'

# Dosyaları tam yollarıyla okuyoruz
train_df = pd.read_csv(os.path.join(klasor_yolu, 'train.csv'))
test_df = pd.read_csv(os.path.join(klasor_yolu, 'test_x.csv'))

print("Başarılı! Veri setleri yüklendi.")
print("Train (Eğitim) Boyutu:", train_df.shape)
print("Test Boyutu:", test_df.shape)

Başarılı! Veri setleri yüklendi.
Train (Eğitim) Boyutu: (10000, 47)
Test Boyutu: (10000, 46)


In [ ]:
# ============================================================
# GÜNCEL PART 3 — GÜVENLİ VE GÜÇLENDİRİLMİŞ 5-FOLD CV EĞİTİMİ
# ============================================================
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import re

# 1. Sütun İsimlerindeki Zararlı Karakterleri Temizleme Fonksiyonu
def lgb_sutun_temizle(df):
    yeni_sutunlar = []
    for col in df.columns:
        # Alfanumerik ve alt çizgi dışındaki her şeyi alt çizgiye çevirir
        clean_name = re.sub(r'[^A-Za-z0-9_]', '_', str(col))
        # Peş peşe gelen alt çizgileri teke indirir
        clean_name = re.sub(r'__+', '_', clean_name)
        yeni_sutunlar.append(clean_name.strip('_'))
    df.columns = yeni_sutunlar
    return df

# 2. Hedef Değişken (y) ve Özelliklerin (X) Hazırlanması
target_col = 'career_success_score'
gereksiz_sutunlar = [target_col, 'id', 'ID', 'Student_ID', 'student_id', 'mentor_feedback_text', 'mentor_feedback_clean']

if target_col in train_df.columns:
    y = train_df[target_col]
else:
    raise ValueError(f"⚠️ '{target_col}' sütunu train_df içinde bulunamadı!")

# Özellik matrislerini oluşturma
X = train_df.drop(columns=[col for col in gereksiz_sutunlar if col in train_df.columns])
X_test = test_df[X.columns]

# Sütun isimlerini LightGBM uyumlu hale getirelim (Hata engelleyici kısım)
X = lgb_sutun_temizle(X)
X_test = lgb_sutun_temizle(X_test)

print(f"📊 Güvenli Eğitim Özellik Boyutu (X): {X.shape}")
print(f"📝 Güvenli Test Özellik Boyutu (X_test): {X_test.shape}")

# 3. Skor Düşürücü Gelişmiş Model Parametreleri
LGBM_OPTIMIZED_PARAMS = {
    "objective"        : "regression",
    "metric"           : "rmse",
    "n_estimators"     : 2500,        # Ağaç sayısını artırarak daha derin öğrenme sağladık
    "learning_rate"    : 0.02,        # Adım büyüklüğünü düşürerek kararlılığı artırdık
    "num_leaves"       : 45,          # Ağaç karmaşıklığını optimize ettik (64 overfitting yapabilir)
    "max_depth"        : -1,
    "min_child_samples": 25,
    "subsample"        : 0.8,
    "colsample_bytree" : 0.7,         # Rastgele sütun seçimi (Overfitting engeller)
    "reg_alpha"        : 0.5,         # L1 Regülasyonu artırıldı
    "reg_lambda"       : 1.5,         # L2 Regülasyonu artırıldı
    "random_state"     : 42,
    "n_jobs"           : -1,
    "verbose"          : -1
}

# 4. 5-Fold Cross Validation Döngüsü
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_tahminler = np.zeros(len(X))
test_tahminler = np.zeros(len(X_test))
fold_mse_listesi = []

print("\n" + "=" * 60)
print(f"🚀 Optimized LightGBM — {N_FOLDS}-Fold CV Eğitim Döngüsü Başladı")
print("=" * 60)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), start=1):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(**LGBM_OPTIMIZED_PARAMS)

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    # Validation ve Test Tahminlerinin Toplanması
    val_pred = model.predict(X_val)
    oof_tahminler[val_idx] = val_pred
    test_tahminler += model.predict(X_test) / N_FOLDS

    fold_mse = mean_squared_error(y_val, val_pred)
    fold_mse_listesi.append(fold_mse)
    print(f"  Fold {fold} | Hata Payı (Val MSE): {fold_mse:.6f}")

# 5. İstatistiksel Sonuçların Yazdırılması
oof_mse = mean_squared_error(y, oof_tahminler)
print("\n" + "=" * 60)
print("🏆 YENİ GÜNCEL DOĞRULAMA SONUÇLARI")
print("=" * 60)
print(f"  Katman Ortalama MSE : {np.mean(fold_mse_listesi):.6f}")
print(f"  Genel OOF MSE Skoru : {oof_mse:.6f}")
print("=" * 60)

# 6. Kaggle Submission Dosyasının Güvenli Oluşturulması
id_sutun_adi = "student_id" if "student_id" in test_df.columns else test_df.columns[0]
submission = pd.DataFrame({
    id_sutun_adi: test_df[id_sutun_adi],
    "career_success_score": test_tahminler
})

# Dosyayı hem Colab'e hem Drive'a kaydedelim
submission.to_csv("submission.csv", index=False)
submission.to_csv("/content/drive/MyDrive/datathon/final_submission_v2.csv", index=False)
print(f"\n✅ 'final_submission_v2.csv' başarıyla üretildi ve Drive'a aktarıldı!")

📊 Güvenli Eğitim Özellik Boyutu (X): (10000, 20054)
📝 Güvenli Test Özellik Boyutu (X_test): (10000, 20054)

🚀 Optimized LightGBM — 5-Fold CV Eğitim Döngüsü Başladı
  Fold 1 | Hata Payı (Val MSE): 85.856591
  Fold 2 | Hata Payı (Val MSE): 87.975660
  Fold 3 | Hata Payı (Val MSE): 77.305806
  Fold 4 | Hata Payı (Val MSE): 78.830264
  Fold 5 | Hata Payı (Val MSE): 87.172672

🏆 YENİ GÜNCEL DOĞRULAMA SONUÇLARI
  Katman Ortalama MSE : 83.428198
  Genel OOF MSE Skoru : 83.428198

✅ 'final_submission_v2.csv' başarıyla üretildi ve Drive'a aktarıldı!


In [ ]:
# ============================================================
# OPTİMİZE EDİLMİŞ PART 3 — GÜÇLENDİRİLMİŞ 5-FOLD CV EĞİTİMİ
# ============================================================
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import re

# 1. Sütun İsimlerini Temizleme
def clean_col_names_advanced(df):
    cols = df.columns
    new_cols = []
    for col in cols:
        new_col = re.sub(r'[^A-z0-9_]', '_', str(col))
        new_col = re.sub(r'__+', '_', new_col)
        new_cols.append(new_col.strip('_'))
    df.columns = new_cols
    return df

target_col = 'career_success_score'
gereksiz_sutunlar = [target_col, 'id', 'ID', 'Student_ID', 'student_id', 'mentor_feedback_text', 'mentor_feedback_clean']

# Ayrıştırma ve sütun temizliği
X = train_df.drop(columns=[col for col in gereksiz_sutunlar if col in train_df.columns])
X_test = test_df[X.columns]

X = clean_col_names_advanced(X)
X_test = clean_col_names_advanced(X_test)

print(f"📊 Eğitim Özellik Boyutu (X): {X.shape}")
print(f"📝 Test Özellik Boyutu (X_test): {X_test.shape}")

# 2. Skor Düşürücü (Optimize Edilmiş) Model Parametreleri
OPTIMIZED_LGBM_PARAMS = {
    "objective"        : "regression",
    "metric"           : "rmse",
    "n_estimators"     : 2500,        # Ağaç sayısı artırıldı
    "learning_rate"    : 0.02,        # Adım büyüklüğü düşürüldü (Daha kararlı)
    "num_leaves"       : 45,          # Karmaşıklık dengelendi
    "max_depth"        : -1,
    "min_child_samples": 25,
    "subsample"        : 0.8,
    "colsample_bytree" : 0.7,         # Her ağaçta sütunların %70'ini rastgele seç (Çeşitlilik sağlar)
    "reg_alpha"        : 0.3,         # L1 Regülasyonu (Ezberleme karşıtı)
    "reg_lambda"       : 1.2,         # L2 Regülasyonu (Ezberleme karşıtı)
    "random_state"     : 42,
    "n_jobs"           : -1,
    "verbose"          : -1
}

# 3. 5-Fold Cross Validation Döngüsü
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_tahminler = np.zeros(len(X))
test_tahminler = np.zeros(len(X_test))
fold_mse_listesi = []

print("\n" + "=" * 60)
print(f"🚀 Gelişmiş LightGBM — 5-Fold Cross Validation")
print("=" * 60)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), start=1):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(**OPTIMIZED_LGBM_PARAMS)

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=-1) # İç logları kapatıp ekranı temiz tutalım
        ]
    )

    # Tahminleri toplama
    val_pred = model.predict(X_val)
    oof_tahminler[val_idx] = val_pred
    test_tahminler += model.predict(X_test) / N_FOLDS

    fold_mse = mean_squared_error(y_val, val_pred)
    fold_mse_listesi.append(fold_mse)
    print(f"  Fold {fold} | Val MSE: {fold_mse:.6f} | Val RMSE: {np.sqrt(fold_mse):.6f}")

# 4. Sonuçlar
oof_mse = mean_squared_error(y, oof_tahminler)
print("\n" + "=" * 60)
# Eski skoru ekranda referans olarak görelim
print(f"🏆 Eski Ortalama MSE: 85.1805")
print(f"🔥 YENİ DOĞRULAMA (OOF) MSE: {oof_mse:.6f}")
print("======================================================")

# 5. Yeni Tahmin Dosyasını Kaydetme
id_sutun_adi = "student_id" if "student_id" in test_df.columns else test_df.columns[0]
submission = pd.DataFrame({
    id_sutun_adi: test_df[id_sutun_adi],
    "career_success_score": test_tahminler
})

submission.to_csv("submission_optimized.csv", index=False)
submission.to_csv("/content/drive/MyDrive/datathon/final_submission_optimized.csv", index=False)
print("\n✅ Yeni ve optimize edilmiş dosya 'final_submission_optimized.csv' adıyla Drive'a kaydedildi!")

📊 Eğitim Özellik Boyutu (X): (10000, 20054)
📝 Test Özellik Boyutu (X_test): (10000, 20054)

🚀 Gelişmiş LightGBM — 5-Fold Cross Validation
  Fold 1 | Val MSE: 86.068415 | Val RMSE: 9.277306
  Fold 2 | Val MSE: 88.284442 | Val RMSE: 9.395980


KeyboardInterrupt: 

In [ ]:
# ============================================================
# HÜCRE 1 — Kütüphaneler
# ============================================================
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

# ============================================================
# HÜCRE 2 — Model parametreleri
# ============================================================
LGBM_PARAMS = {
    "objective"        : "regression",
    "metric"           : "rmse",
    "n_estimators"     : 1000,
    "learning_rate"    : 0.05,
    "num_leaves"       : 64,
    "max_depth"        : -1,
    "min_child_samples": 20,
    "subsample"        : 0.8,
    "colsample_bytree" : 0.8,
    "reg_alpha"        : 0.1,
    "reg_lambda"       : 1.0,
    "random_state"     : 42,
    "n_jobs"           : -1,
    "verbose"          : -1,
}

N_FOLDS   = 5
SEED      = 42
EARLY_STOP= 50   # validation kaybı durduğunda erken dur

# ============================================================
# HÜCRE 3 — 5-Fold CV eğitim döngüsü
# ============================================================
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_tahminler  = np.zeros(len(X))          # Out-of-Fold tahminleri
test_tahminler = np.zeros(len(X_test))     # Her fold tahminlerinin birikimi
fold_mse_listesi = []

print("=" * 60)
print(f"  LightGBM — {N_FOLDS}-Fold Cross Validation")
print("=" * 60)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), start=1):

    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(**LGBM_PARAMS)

    model.fit(
        X_tr, y_tr,
        eval_set              = [(X_val, y_val)],
        callbacks             = [
            lgb.early_stopping(EARLY_STOP, verbose=False),
            lgb.log_evaluation(period=-1),   # fold içi logu kapat
        ],
    )

    # Validation tahmini (OOF)
    val_pred = model.predict(X_val, num_iteration=model.best_iteration_)
    oof_tahminler[val_idx] = val_pred

    # Test tahmini — 5 fold ortalaması için biriktir
    test_tahminler += model.predict(X_test, num_iteration=model.best_iteration_) / N_FOLDS

    fold_mse = mean_squared_error(y_val, val_pred)
    fold_mse_listesi.append(fold_mse)

    print(f"  Fold {fold}  |  Best iter: {model.best_iteration_:4d}"
          f"  |  Val MSE: {fold_mse:.6f}  |  Val RMSE: {np.sqrt(fold_mse):.6f}")

# ============================================================
# HÜCRE 4 — Özet istatistikler
# ============================================================
oof_mse  = mean_squared_error(y, oof_tahminler)
oof_rmse = np.sqrt(oof_mse)

print("\n" + "=" * 60)
print("  Cross Validation Sonuçları")
print("=" * 60)
print(f"  Fold MSE'leri  : {[round(m, 6) for m in fold_mse_listesi]}")
print(f"  Ortalama MSE   : {np.mean(fold_mse_listesi):.6f}"
      f"  ± {np.std(fold_mse_listesi):.6f}")
print(f"  OOF MSE        : {oof_mse:.6f}")
print(f"  OOF RMSE       : {oof_rmse:.6f}")
print("=" * 60)

# ============================================================
# HÜCRE 5 — Kaggle submission dosyası
# ============================================================

# Olası ID sütun isimlerini test_df içinde arayalım
olasi_id_isimleri = ["id", "ID", "Student_ID", "student_id"]
id_sutunu = None

for col in olasi_id_isimleri:
    if col in test_df.columns:
        id_sutunu = col
        break

if id_sutunu is None:
    # Eğer One-Hot Encoding veya işlemler sırasında ID silindiyse,
    # orijinal test dosyasını Drive'dan hızlıca çekip ID'yi oradan alıyoruz
    orijinal_test = pd.read_csv('/content/drive/MyDrive/datathon/test_x.csv')
    id_verisi = orijinal_test.iloc[:, 0] # İlk sütun genelde ID'dir
    id_sutun_adi = orijinal_test.columns[0]
else:
    id_verisi = test_df[id_sutunu]
    id_sutun_adi = id_sutunu

# Yarışma formatına birebir uygun DataFrame oluşturma
submission = pd.DataFrame({
    id_sutun_adi: id_verisi,
    "career_success_score": test_tahminler  # Yarışmanın beklediği tam hedef sütun adı
})

# Dosyayı hem Google Drive'ına hem de yerel Colab hafızasına kaydedelim (Kayıp olmasın)
submission.to_csv("submission.csv", index=False)
submission.to_csv("/content/drive/MyDrive/datathon/final_submission.csv", index=False)

print(f"✅ Dosyalar Başarıyla Kaydedildi!")
print(f"📁 Colab Hafızası: submission.csv")
print(f"📁 Google Drive: /datathon/final_submission.csv")
print(f"📐 Boyut: {submission.shape}")
print("\n📋 İlk 5 Satır Kontrolü:")
print(submission.head())

  LightGBM — 5-Fold Cross Validation
  Fold 1  |  Best iter:  202  |  Val MSE: 88.432624  |  Val RMSE: 9.403862
  Fold 2  |  Best iter:  238  |  Val MSE: 90.393787  |  Val RMSE: 9.507565
  Fold 3  |  Best iter:  213  |  Val MSE: 78.835361  |  Val RMSE: 8.878928
  Fold 4  |  Best iter:  257  |  Val MSE: 79.429309  |  Val RMSE: 8.912312
  Fold 5  |  Best iter:  202  |  Val MSE: 88.811672  |  Val RMSE: 9.423994

  Cross Validation Sonuçları
  Fold MSE'leri  : [88.432624, 90.393787, 78.835361, 79.429309, 88.811672]
  Ortalama MSE   : 85.180551  ± 4.985520
  OOF MSE        : 85.180551
  OOF RMSE       : 9.229331
✅ Dosyalar Başarıyla Kaydedildi!
📁 Colab Hafızası: submission.csv
📁 Google Drive: /datathon/final_submission.csv
📐 Boyut: (10000, 2)

📋 İlk 5 Satır Kontrolü:
   student_id  career_success_score
0  STU_010001             59.432664
1  STU_010002             68.167840
2  STU_010003             72.132729
3  STU_010004             94.597519
4  STU_010005             77.810477


In [ ]:
# ============================================================================
# BTK DATATHON 2026 — FULL PIPELINE (HEDEF: MSE < 80)
# Tek hücre — sıfırdan çalıştır (train.csv ve test_x.csv'yi tekrar okur)
# ============================================================================

import pandas as pd
import numpy as np
import re
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction.text import TfidfVectorizer

# ----------------------------------------------------------------------
# 0) VERİYİ TEMİZ HALİYLE BAŞTAN OKU
# ----------------------------------------------------------------------
klasor_yolu = '/content/drive/MyDrive/datathon'
train_df = pd.read_csv(f'{klasor_yolu}/train.csv')
test_df  = pd.read_csv(f'{klasor_yolu}/test_x.csv')

TARGET = 'career_success_score'
ID_COL = 'student_id'

print(f"Train: {train_df.shape}  |  Test: {test_df.shape}")

# ----------------------------------------------------------------------
# 1) SAYISAL SÜTUNLARDA EKSİK VERİ — MEDYAN İLE DOLDUR (leakage yok)
# ----------------------------------------------------------------------
sayisal_sutunlar = train_df.select_dtypes(include=["number"]).columns.tolist()
if TARGET in sayisal_sutunlar:
    sayisal_sutunlar.remove(TARGET)

for col in sayisal_sutunlar:
    medyan = train_df[col].median()
    train_df[col] = train_df[col].fillna(medyan)
    if col in test_df.columns:
        test_df[col] = test_df[col].fillna(medyan)

# ----------------------------------------------------------------------
# 2) GELİŞMİŞ METİN MÜHENDİSLİĞİ — mentor_feedback_text
# ----------------------------------------------------------------------
def metin_temizle(metin):
    if pd.isnull(metin):
        return ""
    metin = str(metin).lower()
    metin = re.sub(r"[^\w\sçğıöşü!?]", " ", metin)   # noktalama (! ? hariç) sil
    metin = re.sub(r"\d+", " ", metin)
    metin = re.sub(r"\s+", " ", metin)
    return metin.strip()

for df in (train_df, test_df):
    raw = df['mentor_feedback_text'].fillna("")

    # --- Sayısal meta-özellikler (anlam taşır, TF-IDF'ye ek bilgi verir) ---
    df['text_length']        = raw.apply(len)
    df['word_count']         = raw.apply(lambda x: len(x.split()))
    df['exclaim_count']      = raw.apply(lambda x: x.count('!'))
    df['question_count']     = raw.apply(lambda x: x.count('?'))
    df['avg_word_length']    = df.apply(
        lambda r: r['text_length'] / r['word_count'] if r['word_count'] > 0 else 0, axis=1
    )
    df['upper_ratio'] = raw.apply(
        lambda x: sum(1 for c in x if c.isupper()) / len(x) if len(x) > 0 else 0
    )

    # --- Temizlenmiş metin (TF-IDF için) ---
    df['mentor_feedback_clean'] = raw.apply(metin_temizle)

# --- TF-IDF (fit sadece train, ileri n-gram, geniş kelime havuzu) ---
tfidf = TfidfVectorizer(
    max_features  = 4000,
    ngram_range   = (1, 3),
    min_df        = 2,
    max_df        = 0.95,
    sublinear_tf  = True,
    strip_accents = "unicode",
    analyzer      = "word",
)
tfidf.fit(train_df['mentor_feedback_clean'])

train_tfidf = tfidf.transform(train_df['mentor_feedback_clean'])
test_tfidf  = tfidf.transform(test_df['mentor_feedback_clean'])

tfidf_cols = [f"tfidf_{i}" for i in range(train_tfidf.shape[1])]  # nümerik index → garantili temiz isim

train_tfidf_df = pd.DataFrame(train_tfidf.toarray(), columns=tfidf_cols, index=train_df.index)
test_tfidf_df  = pd.DataFrame(test_tfidf.toarray(),  columns=tfidf_cols, index=test_df.index)

train_df = pd.concat([train_df, train_tfidf_df], axis=1)
test_df  = pd.concat([test_df,  test_tfidf_df],  axis=1)

train_df.drop(columns=['mentor_feedback_text', 'mentor_feedback_clean'], inplace=True)
test_df.drop(columns=['mentor_feedback_text', 'mentor_feedback_clean'], inplace=True)

print(f"TF-IDF eklendi → Train: {train_df.shape}  |  Test: {test_df.shape}")

# ----------------------------------------------------------------------
# 3) AKILLI KATEGORİK DÖNÜŞÜM — LightGBM native categorical
# ----------------------------------------------------------------------
kategorik_sutunlar = ['department', 'university_tier', 'target_role',
                       'hobby', 'preferred_social_media_platform']

for col in kategorik_sutunlar:
    train_df[col] = train_df[col].fillna("Bilinmiyor").astype(str)
    test_df[col]  = test_df[col].fillna("Bilinmiyor").astype(str)

    # Aynı kategori kümesini kullan (train + test birleşik)
    tum_kategoriler = pd.Categorical(
        pd.concat([train_df[col], test_df[col]], ignore_index=True)
    ).categories

    train_df[col] = pd.Categorical(train_df[col], categories=tum_kategoriler)
    test_df[col]  = pd.Categorical(test_df[col],  categories=tum_kategoriler)

print("Kategorik sütunlar LightGBM 'category' tipine çevrildi.")

# ----------------------------------------------------------------------
# 4) X / y / X_test HAZIRLA + SÜTUN ADI TEMİZLİĞİ
# ----------------------------------------------------------------------
y = train_df[TARGET]

drop_cols = [TARGET, ID_COL]
X      = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns])
X_test = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns])
X_test = X_test[X.columns]   # sütun sırası garantisi

def clean_col_names(df):
    new_cols = []
    for col in df.columns:
        c = re.sub(r'[^A-Za-z0-9_]', '_', str(col))
        c = re.sub(r'__+', '_', c).strip('_')
        new_cols.append(c)
    df.columns = new_cols
    return df

X      = clean_col_names(X)
X_test = clean_col_names(X_test)

# Kategorik sütunların isimlerini güncel listede yakala
kategorik_sutunlar_clean = [c for c in X.columns
                             if X[c].dtype.name == 'category']

print(f"\nFinal X: {X.shape}  |  X_test: {X_test.shape}")
print(f"Kategorik sütunlar: {kategorik_sutunlar_clean}")

# ----------------------------------------------------------------------
# 5) AGRESİF AMA REGÜLE LIGHTGBM PARAMETRELERİ
# ----------------------------------------------------------------------
LGBM_PARAMS = {
    "objective"        : "regression",
    "metric"           : "rmse",
    "n_estimators"     : 3500,
    "learning_rate"    : 0.015,
    "num_leaves"       : 63,
    "max_depth"        : -1,
    "min_child_samples": 25,
    "subsample"        : 0.8,
    "subsample_freq"   : 1,
    "colsample_bytree" : 0.6,
    "reg_alpha"        : 1.5,
    "reg_lambda"       : 3.0,
    "random_state"     : 42,
    "n_jobs"           : -1,
    "verbose"          : -1,
}

N_FOLDS    = 5
SEED       = 42
EARLY_STOP = 100

# ----------------------------------------------------------------------
# 6) 5-FOLD CV + ENSEMBLE
# ----------------------------------------------------------------------
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_pred  = np.zeros(len(X))
test_pred = np.zeros(len(X_test))
fold_mse_listesi = []

print("\n" + "=" * 60)
print(f"  LightGBM — {N_FOLDS}-Fold CV  (Hedef: MSE < 80)")
print("=" * 60)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y), start=1):

    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(**LGBM_PARAMS)

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        categorical_feature=kategorik_sutunlar_clean,
        callbacks=[
            lgb.early_stopping(EARLY_STOP, verbose=False),
            lgb.log_evaluation(period=-1),
        ],
    )

    val_pred = model.predict(X_val, num_iteration=model.best_iteration_)
    oof_pred[val_idx] = val_pred

    test_pred += model.predict(X_test, num_iteration=model.best_iteration_) / N_FOLDS

    fold_mse = mean_squared_error(y_val, val_pred)
    fold_mse_listesi.append(fold_mse)

    print(f"  Fold {fold}  |  Best iter: {model.best_iteration_:4d}"
          f"  |  Val MSE: {fold_mse:.4f}  |  Val RMSE: {np.sqrt(fold_mse):.4f}")

# ----------------------------------------------------------------------
# 7) ÖZET
# ----------------------------------------------------------------------
oof_mse  = mean_squared_error(y, oof_pred)
oof_rmse = np.sqrt(oof_mse)

print("\n" + "=" * 60)
print("  CV SONUÇLARI")
print("=" * 60)
print(f"  Fold MSE'leri : {[round(m, 4) for m in fold_mse_listesi]}")
print(f"  Ortalama MSE  : {np.mean(fold_mse_listesi):.4f} ± {np.std(fold_mse_listesi):.4f}")
print(f"  OOF MSE       : {oof_mse:.4f}")
print(f"  OOF RMSE      : {oof_rmse:.4f}")
print("=" * 60)

# ----------------------------------------------------------------------
# 8) SUBMISSION — NaN KONTROLÜ + KAYIT
# ----------------------------------------------------------------------
assert not np.isnan(test_pred).any(), "❌ Tahminlerde NaN var!"
assert test_df[ID_COL].isnull().sum() == 0, "❌ ID sütununda NaN var!"

submission = pd.DataFrame({
    ID_COL : test_df[ID_COL],
    TARGET : test_pred
})

assert submission.isnull().sum().sum() == 0, "❌ Submission'da NaN var!"
assert submission.shape == (len(test_df), 2), "❌ Boyut hatalı!"

cikis_yolu = f'{klasor_yolu}/final_submission_target_80.csv'
submission.to_csv(cikis_yolu, index=False)
submission.to_csv("final_submission_target_80.csv", index=False)

print(f"\n✅ Kaydedildi: {cikis_yolu}")
print(f"📐 Boyut: {submission.shape}")
print("\n📋 İlk 5 satır:")
print(submission.head())

Train: (10000, 47)  |  Test: (10000, 46)
TF-IDF eklendi → Train: (10000, 4052)  |  Test: (10000, 4051)
Kategorik sütunlar LightGBM 'category' tipine çevrildi.

Final X: (10000, 4050)  |  X_test: (10000, 4050)
Kategorik sütunlar: ['department', 'university_tier', 'target_role', 'hobby', 'preferred_social_media_platform']

  LightGBM — 5-Fold CV  (Hedef: MSE < 80)
  Fold 1  |  Best iter:  683  |  Val MSE: 82.8887  |  Val RMSE: 9.1043
  Fold 2  |  Best iter:  774  |  Val MSE: 84.9622  |  Val RMSE: 9.2175
  Fold 3  |  Best iter:  680  |  Val MSE: 73.7310  |  Val RMSE: 8.5867
  Fold 4  |  Best iter: 1162  |  Val MSE: 74.5468  |  Val RMSE: 8.6341
  Fold 5  |  Best iter:  813  |  Val MSE: 83.3124  |  Val RMSE: 9.1276

  CV SONUÇLARI
  Fold MSE'leri : [82.8887, 84.9622, 73.731, 74.5468, 83.3124]
  Ortalama MSE  : 79.8882 ± 4.7522
  OOF MSE       : 79.8882
  OOF RMSE      : 8.9380

✅ Kaydedildi: /content/drive/MyDrive/datathon/final_submission_target_80.csv
📐 Boyut: (10000, 2)

📋 İlk 5 satır:
 

In [ ]:
# ============================================================
# YENİ HÜCRE 1 — Hangi özellikler işe yarıyor?
# ============================================================
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("🏆 İlk 30 özellik:")
print(importance_df.head(30).to_string(index=False))

tfidf_importance = importance_df[importance_df['feature'].str.startswith('tfidf_')]['importance'].sum()
toplam = importance_df['importance'].sum()
print(f"\n📊 TF-IDF toplam katkı: {100*tfidf_importance/toplam:.1f}%")
print(f"🪦 Sıfır importance'lı sütun sayısı: {(importance_df['importance']==0).sum()}")

🏆 İlk 30 özellik:
                   feature  importance
 technical_interview_score        1532
       communication_score        1368
     project_quality_score        1293
           portfolio_score        1107
          cv_quality_score         961
                      cgpa         957
            teamwork_score         913
        presentation_score         904
               cloud_score         873
        hr_interview_score         842
     problem_solving_score         833
          github_avg_stars         789
               target_role         788
              coding_score         770
    linkedin_profile_score         755
              devops_score         753
             backend_score         669
         applications_sent         631
        english_exam_score         620
          application_year         605
           avg_word_length         596
    machine_learning_score         595
         github_repo_count         590
           graduation_year         586
       

In [ ]:
# ============================================================
# YENİ HÜCRE 2 — 3 farklı seed ile eğit, tahminleri ortala
# ============================================================
SEEDS = [42, 123, 2024]
test_pred_multi = np.zeros(len(X_test))
oof_pred_multi  = np.zeros(len(X))

for seed in SEEDS:
    LGBM_PARAMS["random_state"] = seed
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

    oof_temp  = np.zeros(len(X))
    test_temp = np.zeros(len(X_test))

    for tr_idx, val_idx in kf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        m = lgb.LGBMRegressor(**LGBM_PARAMS)
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              categorical_feature=kategorik_sutunlar_clean,
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(period=-1)])

        oof_temp[val_idx] = m.predict(X_val, num_iteration=m.best_iteration_)
        test_temp += m.predict(X_test, num_iteration=m.best_iteration_) / N_FOLDS

    seed_mse = mean_squared_error(y, oof_temp)
    print(f"Seed {seed}: OOF MSE = {seed_mse:.4f}")

    oof_pred_multi  += oof_temp / len(SEEDS)
    test_pred_multi += test_temp / len(SEEDS)

print(f"\n🎯 Multi-seed OOF MSE: {mean_squared_error(y, oof_pred_multi):.4f}")

Seed 42: OOF MSE = 79.8882
Seed 123: OOF MSE = 80.2175
Seed 2024: OOF MSE = 80.4128

🎯 Multi-seed OOF MSE: 79.1498


In [ ]:
# ============================================================
# YENİ HÜCRE 2 — 3 farklı seed ile eğit, tahminleri ortala
# ============================================================
SEEDS = [42, 123, 2024]
test_pred_multi = np.zeros(len(X_test))
oof_pred_multi  = np.zeros(len(X))

for seed in SEEDS:
    LGBM_PARAMS["random_state"] = seed
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

    oof_temp  = np.zeros(len(X))
    test_temp = np.zeros(len(X_test))

    for tr_idx, val_idx in kf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        m = lgb.LGBMRegressor(**LGBM_PARAMS)
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              categorical_feature=kategorik_sutunlar_clean,
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(period=-1)])

        oof_temp[val_idx] = m.predict(X_val, num_iteration=m.best_iteration_)
        test_temp += m.predict(X_test, num_iteration=m.best_iteration_) / N_FOLDS

    seed_mse = mean_squared_error(y, oof_temp)
    print(f"Seed {seed}: OOF MSE = {seed_mse:.4f}")

    oof_pred_multi  += oof_temp / len(SEEDS)
    test_pred_multi += test_temp / len(SEEDS)

print(f"\n🎯 Multi-seed OOF MSE: {mean_squared_error(y, oof_pred_multi):.4f}")

Seed 42: OOF MSE = 79.8882
Seed 123: OOF MSE = 80.2175
Seed 2024: OOF MSE = 80.4128

🎯 Multi-seed OOF MSE: 79.1498


In [ ]:
# ============================================================
# YENİ HÜCRE 3 — Seed ensemble tahminleriyle submission kaydet
# ============================================================
assert not np.isnan(test_pred_multi).any(), "❌ NaN var!"

submission = pd.DataFrame({
    ID_COL : test_df[ID_COL],
    TARGET : test_pred_multi
})

assert submission.isnull().sum().sum() == 0
assert submission.shape == (len(test_df), 2)

submission.to_csv(f'{klasor_yolu}/final_submission_target_80.csv', index=False)
submission.to_csv("final_submission_target_80.csv", index=False)

print("✅ Final submission (seed ensemble) kaydedildi.")
print(f"Yeni OOF MSE: {mean_squared_error(y, oof_pred_multi):.4f}")
print(submission.head())

✅ Final submission (seed ensemble) kaydedildi.
Yeni OOF MSE: 79.1498
   student_id  career_success_score
0  STU_010001             59.844221
1  STU_010002             73.192177
2  STU_010003             74.900559
3  STU_010004             95.688921
4  STU_010005             74.710637


In [ ]:
sample_sub = pd.read_csv(f'{klasor_yolu}/sample_submission.csv')

print("Sample submission boyutu:", sample_sub.shape)
print("Senin submission boyutu :", submission.shape)

print("\nSample sub ilk 5 ID:")
print(sample_sub.iloc[:, 0].head().tolist())

print("\nSenin submission ilk 5 ID:")
print(submission['student_id'].head().tolist())

print("\nID sırası TAMAMEN aynı mı:")
# Compare only up to the length of the smaller DataFrame (sample_sub)
print((sample_sub.iloc[:, 0].values == submission['student_id'].head(len(sample_sub)).values).all())

print("\nSample sub sütun isimleri:", sample_sub.columns.tolist())
print("Senin sütun isimlerin   :", submission.columns.tolist())

Sample submission boyutu: (2, 2)
Senin submission boyutu : (10000, 2)

Sample sub ilk 5 ID:
['STU_010001', 'STU_010002']

Senin submission ilk 5 ID:
['STU_010001', 'STU_010002', 'STU_010003', 'STU_010004', 'STU_010005']

ID sırası TAMAMEN aynı mı:
True

Sample sub sütun isimleri: ['student_id', 'career_success_score']
Senin sütun isimlerin   : ['student_id', 'career_success_score']


In [ ]:
# 1) Tahmin dağılımı sağlıklı mı?
print("y (train target) istatistikleri:")
print(y.describe())

print("\ntest_pred_multi istatistikleri:")
print(pd.Series(test_pred_multi).describe())

# 2) NaN/Inf kontrolü
print("\nNaN sayısı:", np.isnan(test_pred_multi).sum())
print("Inf sayısı:", np.isinf(test_pred_multi).sum())

# 3) Sütun sırası train/test birebir aynı mı (isim bazında)
print("\nSütun sırası aynı mı:", list(X.columns) == list(X_test.columns))
print("Toplam sütun sayısı X:", X.shape[1], " X_test:", X_test.shape[1])

# 4) TF-IDF sütunları test'te de anlamlı dolu mu?
tfidf_cols_check = [c for c in X.columns if c.startswith('tfidf_')]
print("\nTrain TF-IDF toplam değer:", X[tfidf_cols_check].values.sum())
print("Test  TF-IDF toplam değer:", X_test[tfidf_cols_check].values.sum())

# 5) Kategorik kodlama train/test'te aynı mı?
for col in kategorik_sutunlar_clean:
    print(f"\n{col}:")
    print("  Train kategoriler:", X[col].cat.categories.tolist())
    print("  Test  kategoriler:", X_test[col].cat.categories.tolist())

y (train target) istatistikleri:
count    10000.000000
mean        76.942507
std         15.186669
min          0.000000
25%         66.937500
50%         77.810000
75%         88.472500
max        100.000000
Name: career_success_score, dtype: float64

test_pred_multi istatistikleri:
count    10000.000000
mean        76.192826
std         12.273606
min         35.066878
25%         67.595289
50%         76.049646
75%         85.164364
max        106.093664
dtype: float64

NaN sayısı: 0
Inf sayısı: 0

Sütun sırası aynı mı: True
Toplam sütun sayısı X: 4050  X_test: 4050

Train TF-IDF toplam değer: 69554.54110660226
Test  TF-IDF toplam değer: 69424.14724184904

department:
  Train kategoriler: ['Computer Engineering', 'Electrical Electronics Engineering', 'Industrial Engineering', 'Management Information Systems', 'Mathematics', 'Software Engineering', 'Statistics']
  Test  kategoriler: ['Computer Engineering', 'Electrical Electronics Engineering', 'Industrial Engineering', 'Management In